# 💬 شام — مرحلة التجميع والمحادثة (المرحلة الثالثة)

هنا يصبح شام **نموذجاً واحداً كاملاً** يجيب بالنص والصورة والصوت والفيديو، ويبحث بنفسه عند الحاجة. كل تشغيل:

1. **الدمج المحروس للمعرفة**: يجمع ما تعلّمته كل مسارات شام (المرحلة الأولى، Track A، Track B البحث الذاتي، وأحدث المرحلة الثانية) في هذا الدماغ. يُقبل الدمج فقط إن تحسّنت كل المهارات معاً، وإلا تُعاد الأوزان كما كانت. ويدخل نص Track B المجموع من الإنترنت في «التذكّر».
2. **صيغة حوار شام**: `<BOS> <USER> سؤال <SHAM> جواب <EOS>`، فيتعلّم الإجابة ومتى يتوقف.
3. **كل الوسائط داخل المحادثة**:
   - الصورة: «صف هذه الصورة» ↔ «ارسم: …»
   - الصوت: «اكتب ما يقال» ↔ «انطق: …»
   - **الفيديو بصوته**: «صف هذا الفيديو» ↔ «أنشئ فيديو: …» (لأول مرة يتعلّم شام الفيديو فعلاً).
4. **البحث المتعلَّم**: يكتب شام بنفسه `<SEARCH_START> استعلام <SEARCH_END>`، فتصله النتيجة ويكمل منها. يتعلّم ذلك من ويكيبيديا العربية دون أي تسميات يدوية.
5. **التذكّر**: نص عربي ممزوج بالتدريب، حتى لا ينسى ما تعلّمه.
6. **المكافأة الذاتية المرتكزة**: يكتب عدة إجابات، ويتعلّم تفضيل أفضلها على أسوئها. تُقبل الجولة فقط إن لم يسؤ على أسئلة لم يرها.
7. **المقياس الثابت** قبل الجلسة وبعدها، ويُرسل إلى تيليجرام.

البيانات: `arbml/CIDAR` و`CohereLabs/aya_dataset` العربية (apache-2.0، مكتوبة بأيدٍ بشرية)، وويكيبيديا العربية، وصور وأصوات ومقاطع Kinetics المرخّصة بالمشاع الإبداعي فقط. كل تشغيل يأخذ بيانات **جديدة** ويكمل بعد آخر موضع.

**الإعداد:** Accelerator: أي خيار (GPU أو None). Internet: On. Secrets: `GITHUB_TOKEN`، `KAGGLE_USERNAME`، `KAGGLE_KEY` (واختيارياً `TELEGRAM_BOT_TOKEN` و`TELEGRAM_CHAT_ID`).

### 1) سحب كود شام من GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


In [ ]:
try:
    import tokenizers
    print(f"مكتبة tokenizers متوفرة مسبقاً (نسخة {tokenizers.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
    print("تم تثبيت مكتبة tokenizers.")

try:
    import soundfile
    print("مكتبة soundfile متوفرة مسبقاً.")
except ImportError:
    subprocess.run(["pip", "install", "-q", "soundfile"], check=True)
    print("تم تثبيت مكتبة soundfile.")


### 2) تحميل آخر نقطة لشام مع أداة تقسيم النص وأداتي الصورة والصوت المحفوظة معها

In [ ]:
import time
SESSION_START = time.time()
import torch
from pathlib import Path
from checkpoint import load_checkpoint
from text_tokenizer import ShamTextTokenizer
from sham_inputs import fetch_dataset
from tokenizer_select import select_pretrained_tokenizer
from train_image_tokenizer import save_tokenizer_checkpoint as save_image_tokenizer
from train_audio_tokenizer import save_tokenizer_checkpoint as save_audio_tokenizer
from sham_chat import load_progress

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"الجهاز: {device}")

CHAT_OWN = fetch_dataset("sham-chat-checkpoint")
_ck = sorted(CHAT_OWN.rglob("final_chat.pt")) if CHAT_OWN else []
if _ck:
    base_ckpt, BASE_DIR = _ck[0], _ck[0].parent
    print("استئناف مرحلة المحادثة من ناتجها السابق.")
else:
    CHAT_OWN = None
    _mm = fetch_dataset("sham-multimodal-checkpoint")
    _ck = sorted(_mm.rglob("final_" + "multimodal.pt")) if _mm else []
    assert _ck, "لم يُنشر ناتج المرحلة الثانية (sham-multimodal-checkpoint) بعد — شغّلي دفتر المرحلة الثانية أولاً."
    base_ckpt, BASE_DIR = _ck[0], _ck[0].parent
    print("بداية مرحلة المحادثة من ناتج المرحلة الثانية.")

model, start_step, _ = load_checkpoint(base_ckpt, map_location=device)
tokenizer = ShamTextTokenizer.load(str(sorted(BASE_DIR.rglob("sham_small_tokenizer.json"))[0]))
tokenizer.save("/kaggle/working/sham_small_tokenizer.json")
print(f"نقطة الانطلاق: {base_ckpt.name} — الخطوة {start_step:,}، vocab={tokenizer.vocab_size:,}")

progress = load_progress([CHAT_OWN]) if CHAT_OWN else {}
progress.setdefault("merged", {})
if not CHAT_OWN:  # نقطة الانطلاق نفسها لا تُدمج مع نفسها
    progress["merged"]["sham-multimodal-checkpoint"] = start_step

# نفس أداتي الترميز اللتين تدرّب بهما النموذج (المحفوظتين مع النقطة نفسها).
_img, _aud = select_pretrained_tokenizer("image", BASE_DIR), select_pretrained_tokenizer("audio", BASE_DIR)
assert _img and _aud, f"أداتا الصورة/الصوت غير موجودتين بجانب {base_ckpt.name} — أعيدي تشغيل المرحلة الثانية لتنشرهما."
(image_tokenizer, _img_step, _), (audio_tokenizer, _aud_step, _) = _img, _aud
image_tokenizer, audio_tokenizer = image_tokenizer.to("cpu").eval(), audio_tokenizer.to("cpu").eval()
save_image_tokenizer("/kaggle/working/image_tokenizer.pt", image_tokenizer, step=_img_step)
save_audio_tokenizer("/kaggle/working/audio_tokenizer.pt", audio_tokenizer, step=_aud_step)
from sham_media_link import confirm_saved_tokenizers
confirm_saved_tokenizers(image_tokenizer, audio_tokenizer, BASE_DIR)

### 3) بيانات هذه الجلسة: حوار + صورة + صوت + فيديو + بحث + نص للتذكّر (مع عيّنة تحقق لا يُدرَّب عليها)

In [ ]:
import json, random
import soundfile as sf
from PIL import Image
from model import image_token_id_to_vocab_id, audio_token_id_to_vocab_id
from mel_spectrogram import waveform_to_mel_spectrogram
from sham_chat import (load_arabic_dialogues, build_chat_example, build_media_chat_examples, build_video_chat_examples,
                       build_search_example, load_wiki_articles, split_holdout, batch_examples,
                       text_replay_batches, interleave)
from sham_merge import research_corpus_files
from sham_data_sources import Ledger, collect
from dataset import ImageCaptionDataset, AudioTranscriptDataset, MultimodalCollator, AudioMultimodalCollator

MAX_DIALOGUES = 4_000      # أمثلة حوار جديدة في كل جلسة
MAX_MEDIA = 600            # صور + مقاطع صوت (من كل نوع)
MAX_VIDEOS = 60            # مقاطع فيديو بصوتها (Kinetics المرخّصة CC فقط)
VIDEO_KEYFRAMES = 2        # إطارات كل فيديو داخل سياق النموذج (يطابق ما يطلبه البوت)
MAX_SEARCH = 1_500         # أمثلة «ابحث ثم أجب» من ويكيبيديا
CHAT_BATCH = 8
MAX_LEN = min(1024, model.cfg.max_seq_len)
rng = random.Random(start_step)
Path("/kaggle/working/checkpoints").mkdir(parents=True, exist_ok=True)

# — الحوار
dialogues, progress = load_arabic_dialogues(progress, MAX_DIALOGUES)
if not dialogues:  # انتهت كل الأمثلة المتاحة: دورة جديدة عليها
    progress.update({"cidar": 0, "aya_ar": 0, "epoch": int(progress.get("epoch", 0)) + 1})
    dialogues, progress = load_arabic_dialogues(progress, MAX_DIALOGUES)
train_dialogues, holdout_dialogues = split_holdout(dialogues, seed=start_step)
print(f"حوار: {len(train_dialogues):,} للتدريب + {len(holdout_dialogues):,} للتحقق (الدورة {progress.get('epoch', 0)})")
chat_examples = [build_chat_example(tokenizer.encode(q), tokenizer.encode(a), max_len=MAX_LEN) for q, a in train_dialogues]
holdout_chat = [build_chat_example(tokenizer.encode(q), tokenizer.encode(a), max_len=MAX_LEN) for q, a in holdout_dialogues]
# — التصحيح الذاتي المتعلَّم: مسودة معطوبة بأخطاء شام نفسها ← الإجابة الصحيحة
from sham_correct import correction_examples
correct_examples = correction_examples(tokenizer.encode, train_dialogues, share=0.25, seed=start_step, max_len=MAX_LEN)
print(f"أمثلة التصحيح الذاتي: {len(correct_examples):,}")

# — الصورة والصوت داخل المحادثة
media_examples = []
from sham_media_link import collect_fixed_pairs, data_report
chat_ledgers = {k: Ledger.load(k, name=f"chat_{k}") for k in ("image", "audio")}
FIXED_PAIRS = collect_fixed_pairs(("image", "audio"), chat_ledgers, image_size=image_tokenizer.cfg.image_size)
for kind in ("image", "audio"):
    ledger = chat_ledgers[kind]
    manifest, _stats = collect(kind, f"/kaggle/working/corpus/{kind}", MAX_MEDIA, ledger, image_size=image_tokenizer.cfg.image_size)
    ledger.save("/kaggle/working/checkpoints")
    if kind == "image":
        ds = ImageCaptionDataset(manifest, image_size=image_tokenizer.cfg.image_size)
        encode = MultimodalCollator(tokenizer, image_tokenizer).encode_batch_images
    else:
        ds = AudioTranscriptDataset(manifest, n_mels=audio_tokenizer.cfg.n_mels, segment_frames=audio_tokenizer.cfg.segment_frames)
        encode = AudioMultimodalCollator(tokenizer, audio_tokenizer).encode_batch_audio
    data_report("الصورة" if kind == "image" else "الصوت", _stats, ds)
    for s in range(0, len(ds), 16):
        items = [ds[i] for i in range(s, min(s + 16, len(ds)))]
        with torch.no_grad():
            ids = encode(items)
        for (caption, _), media in zip(items, ids):
            media_examples += build_media_chat_examples(tokenizer.encode, caption, media.tolist(), kind, rng)
media_examples, holdout_media = split_holdout(media_examples, seed=start_step + 1)
print(f"صورة/صوت داخل المحادثة: {len(media_examples):,} (+ {len(holdout_media)} للتحقق)")

# — الفيديو بصوته (لأول مرة داخل النموذج نفسه)
def _frame_tensor(path):
    size = image_tokenizer.cfg.image_size
    im = Image.open(path).convert("RGB").resize((size, size))
    return torch.tensor(list(im.getdata()), dtype=torch.float32).view(size, size, 3).permute(2, 0, 1) / 127.5 - 1.0

video_examples = []
try:
    video_ledger = Ledger.load("video", name="chat_video")
    video_manifest, _ = collect("video", "/kaggle/working/corpus/videos", MAX_VIDEOS, video_ledger,
                                image_size=image_tokenizer.cfg.image_size, num_frames=8)
    video_ledger.save("/kaggle/working/checkpoints")
    vroot = Path(video_manifest).parent
    for line in open(video_manifest, encoding="utf-8"):
        rec = json.loads(line)
        paths = rec["frames"]
        picks = [paths[round(i * (len(paths) - 1) / max(VIDEO_KEYFRAMES - 1, 1))] for i in range(VIDEO_KEYFRAMES)]
        with torch.no_grad():
            frames = torch.stack([_frame_tensor(vroot / p) for p in picks])
            grids = image_tokenizer.encode(frames)
            frame_ids = image_token_id_to_vocab_id(grids.view(len(picks), -1)).tolist()
            audio_ids = None
            if rec.get("audio"):
                wave, sr = sf.read(str(vroot / rec["audio"]), dtype="float32", always_2d=False)
                wave = torch.from_numpy(wave.mean(axis=1) if wave.ndim > 1 else wave)
                mel = waveform_to_mel_spectrogram(wave[: audio_tokenizer.cfg.segment_frames * 160], sr,
                                                  audio_tokenizer.cfg.n_mels, audio_tokenizer.cfg.segment_frames)
                audio_ids = audio_token_id_to_vocab_id(audio_tokenizer.encode(mel.unsqueeze(0)).view(1, -1))[0].tolist()
        video_examples += build_video_chat_examples(tokenizer.encode, rec["caption"], frame_ids, audio_ids, rng)
except Exception as exc:  # الفيديو لا يوقف بقية الجلسة
    print(f"⚠ تعذّر جمع الفيديو هذه الجلسة: {exc}")
print(f"فيديو داخل المحادثة: {len(video_examples):,}")

# — البحث المتعلَّم (ابحث ثم أجب) من ويكيبيديا العربية
search_examples = []
try:
    _skip = int(progress.get("wiki_search", 0))
    for title, text in load_wiki_articles(MAX_SEARCH, _skip):
        ex = build_search_example(tokenizer.encode, title, text, rng)
        if ex and len(ex[0]) <= MAX_LEN:
            search_examples.append(ex)
    progress["wiki_search"] = _skip + MAX_SEARCH
except Exception as exc:
    print(f"⚠ تعذّر تجهيز أمثلة البحث: {exc}")
print(f"أمثلة «ابحث ثم أجب»: {len(search_examples):,}")

# — التذكّر: ويكيبيديا + نص Track B المجموع من الإنترنت (دمج المعرفة على مستوى البيانات)
_research = fetch_dataset("sham-research-track-checkpoint-v2")
extra_text = research_corpus_files(_research)
print(f"نص البحث الذاتي (Track B) المضاف للتذكّر: {len(extra_text)} ملف")
chat_batches = batch_examples(chat_examples, CHAT_BATCH, seed=start_step)
text_batches = text_replay_batches(tokenizer, min(512, MAX_LEN), 4, max(len(chat_batches) // 2, 1) + 4,
                                   "/kaggle/working/replay_text", max_position=20_000, seed=start_step,
                                   extra_files=extra_text)
holdout_text, text_batches = text_batches[:4], text_batches[4:]
media_batches = batch_examples(media_examples, 4, seed=start_step + 1)
video_batches = batch_examples(video_examples, 2, seed=start_step + 2)
search_batches = batch_examples(search_examples, 4, seed=start_step + 3)
correct_batches = batch_examples(correct_examples, CHAT_BATCH, seed=start_step + 4)
all_batches = interleave([chat_batches, media_batches, video_batches, search_batches, text_batches, correct_batches],
                         [1.0, 0.6, 0.3, 0.4, 0.5, 0.3], seed=start_step)
# ميزان الحارس: عيّنات لم يُدرَّب عليها من كل مهارة (محدودة الحجم ليبقى سريعاً)
GATE = {"chat": batch_examples(holdout_chat, 8)[:8], "media": batch_examples(holdout_media, 4)[:6], "text": holdout_text}
GATE = {k: v for k, v in GATE.items() if v}
print(f"دفعات: حوار {len(chat_batches):,} + صورة/صوت {len(media_batches):,} + فيديو {len(video_batches):,} "
      f"+ بحث {len(search_batches):,} + تذكّر {len(text_batches):,} + تصحيح {len(correct_batches):,} = {len(all_batches):,}")
assert all_batches, "لا توجد بيانات لهذه الجلسة."

### 4) المقياس الثابت — قبل الجلسة

In [ ]:
from sham_eval import evaluate, fixed_eval_pairs, format_eval

EVAL_PAIRS = fixed_eval_pairs()
model.to(device)
eval_before = evaluate(model, tokenizer, device, chat=CHAT_OWN is not None, pairs=EVAL_PAIRS)
print(format_eval("قبل الجلسة", start_step, eval_before))
from sham_media_link import measure_link, link_report
_media_toks = {"image": image_tokenizer, "audio": audio_tokenizer}
link_before = measure_link(model, tokenizer, _media_toks, FIXED_PAIRS, device, chat=True)
link_report(link_before)

### 5) الدمج المحروس للمعرفة

يُجرَّب كل مسار بنسب مختلفة، ويُقبل فقط إن تحسّنت **كل** المهارات (حوار، صورة/صوت، نص) على بيانات لم يُدرَّب عليها. كل نسخة من كل مسار تُدمج مرة واحدة فقط.

In [ ]:
from sham_merge import KNOWN_SOURCES, load_source, guarded_merge, batches_loss

def gate_scores(m):
    return {k: batches_loss(m, v, device) for k, v in GATE.items()}

merge_sources = []
merge_report = []
for _name, _pattern, _rows in KNOWN_SOURCES:
    _root = fetch_dataset(_name)
    if not _root:
        continue
    _loaded, _why = load_source(_root, _pattern, model, tokenizer)
    if _loaded is None:
        merge_report.append(f"⏭ {_name}: {_why}")
        continue
    _other, _step = _loaded
    if progress["merged"].get(_name, -1) >= _step:
        merge_report.append(f"⏭ {_name}: الخطوة {_step:,} جُرّبت سابقاً")
        continue
    merge_sources.append((f"{_name} (خطوة {_step:,})", _other.to(device), _rows))
    progress["merged"][_name] = _step

if merge_sources:
    merge_report += guarded_merge(model, merge_sources, gate_scores)
del merge_sources
print("\n".join(merge_report) or "لا مسارات جديدة للدمج هذه الجلسة.")

### 6) التدريب (على أي معالج تم اختياره، ضمن وقت الجلسة)

In [ ]:
from train import TrainConfig, train, build_optimizer

GRAD_ACCUM = 2
SESSION_HOURS = 5.0 if device.startswith("cuda") else 11.0   # حد الجلسة (GPU أقصر حفاظاً على الرصيد)
RESERVE_SECONDS = 50 * 60                                     # للمكافأة الذاتية والمقياس والنشر
optimizer = build_optimizer(model, lr=5e-5, weight_decay=0.1)

t0 = time.time()
model.train()
for _b in all_batches[:5]:  # قياس السرعة الحقيقية على هذا الجهاز (لا تحديث للأوزان)
    ids, labels = _b if isinstance(_b, tuple) else (_b, _b)
    _, loss = model(ids.to(device), labels=labels.to(device))
    loss.backward()
optimizer.zero_grad()
per_batch = max((time.time() - t0) / min(5, len(all_batches)), 1e-3)
train_seconds = max(600, SESSION_HOURS * 3600 - (time.time() - SESSION_START) - RESERVE_SECONDS)
session_batches = all_batches[: int(train_seconds * 0.9 / per_batch)]
TOTAL_STEPS = max(len(session_batches) // GRAD_ACCUM, 1)
print(f"~{per_batch:.2f} ث/دفعة على {device} — {len(session_batches):,} دفعة = {TOTAL_STEPS:,} خطوة تحديث.")

train_cfg = TrainConfig(
    seq_len=MAX_LEN, batch_size=CHAT_BATCH, grad_accum_steps=GRAD_ACCUM, lr=5e-5,
    warmup_steps=max(10, TOTAL_STEPS // 50), total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints", checkpoint_every=200, log_every=20,
    max_wall_clock_seconds=train_seconds,
)
loss_history = train(model, session_batches, train_cfg, device=device, start_step=start_step, resume_optimizer=optimizer)
final_step = start_step + len(loss_history)
print(f"انتهى: {len(loss_history):,} خطوة — الخطوة النهائية {final_step:,}")

### 7) المكافأة الذاتية المرتكزة (تُقبل فقط إن لم يسؤ على أسئلة لم يرها)

In [ ]:
from sham_reward import guarded_self_reward

REWARD_QUESTIONS = 48
_qa = random.Random(final_step).sample(train_dialogues, min(REWARD_QUESTIONS, len(train_dialogues)))
reward_report = guarded_self_reward(model, tokenizer, _qa, device,
                                    gate_fn=lambda m: sum(gate_scores(m).values()))
print(reward_report)

# الاستدلال المتحقَّق (GRPO): يقيس أولاً، ويبقى نائماً حتى تتجاوز إصابة شام العتبة
from sham_grpo import guarded_grpo
grpo_report = guarded_grpo(model, tokenizer, device, gate_fn=lambda m: sum(gate_scores(m).values()), seed=final_step)
print(grpo_report)

# هل تعلّم شام إصلاح مسودته؟ (خسارة الإجابة بلا مسودة مقابل بعد مسودة معطوبة)
from sham_correct import revision_gain, format_gain
correct_gain = revision_gain(model, tokenizer.encode, holdout_dialogues[:32], device, seed=final_step)
correct_report = format_gain(correct_gain)
print(correct_report)

### 8) المقياس الثابت — بعد الجلسة، وتقرير تيليجرام

In [ ]:
eval_after = evaluate(model, tokenizer, device, chat=True, pairs=EVAL_PAIRS)
link_summary = link_report(link_before, measure_link(model, tokenizer, _media_toks, FIXED_PAIRS, device, chat=True))
report = "\n".join([format_eval("مرحلة التجميع والمحادثة", final_step, eval_after, before=eval_before),
                    link_summary, "🧩 دمج المعرفة:", *(merge_report or ["لا جديد"]), reward_report, grpo_report, correct_report])
print(report)
try:
    from kaggle_secrets import UserSecretsClient
    _tg_token = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
    _tg_chat = UserSecretsClient().get_secret("TELEGRAM_CHAT_ID")
except Exception:
    _tg_token, _tg_chat = None, None
from telegram_report import send_telegram_message
send_telegram_message(_tg_token, _tg_chat, report[:4000])

### 9) الحفظ والنشر إلى `sham-chat-checkpoint`

In [ ]:
import shutil
from checkpoint import save_checkpoint
from sham_inputs import publish_dataset

save_checkpoint("/kaggle/working/checkpoints/final_chat.pt", model, final_step)
upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
shutil.copy2("/kaggle/working/checkpoints/final_chat.pt", upload_dir / "final_chat.pt")
for _f in ("sham_small_tokenizer.json", "image_tokenizer.pt", "audio_tokenizer.pt"):
    shutil.copy2(Path("/kaggle/working") / _f, upload_dir / _f)
for _ledger in Path("/kaggle/working/checkpoints").glob("*_ledger.json"):
    shutil.copy2(_ledger, upload_dir / _ledger.name)
(upload_dir / "chat_progress.json").write_text(json.dumps(progress), encoding="utf-8")
(upload_dir / "eval_history.json").write_text(json.dumps(
    {"step": final_step, "before": {k: v for k, v in eval_before.items() if k != "samples"},
     "after": {k: v for k, v in eval_after.items() if k != "samples"},
     "merge": merge_report, "reward": reward_report, "grpo": grpo_report, "correct": correct_gain}, ensure_ascii=False), encoding="utf-8")
publish_dataset(upload_dir, "sham-chat-checkpoint", f"chat stage at step {final_step:,}")